# 05 — Similarity facts

Turn each DayHike in one region into a row of similarity facts, one Facet at a
time, and write that table next to the Catalog. `05a_similarity.ipynb` compares
DayHikes from it. This is the notebook design phase of ADR-0011 (proposed):
production Ranking, the CLI and the Catalog do not change.

**Region:** DayHikes whose Trailhead lies in lat 42.3–46.9, lon 13.3–19.5
(Slovenia + Croatia: Julian Alps, karst, Velebit, the coast, the Pannonian
lowland). `load_region` reads the GeometryStore the way `04a_geometry.ipynb`
does (vertices, amenities, dayhikes, ways as GeoDataFrames) with the Catalog's
`feature_vectors`, Trailhead and Difficulty merged onto `dayhikes`. Rows are
filtered by region DayHike id in SQL; the full GeometryStore is ~19 GB.

Open `data/catalog.sqlite` plus `data/catalog.geometry.sqlite` when both exist.
Otherwise (and always under pytest) build the committed OSM fixture
`tests/fixtures/catalog.osm`, whose DayHikes all lie inside the same box.

**Facets here:** Effort, Altitude, Profile, Landscape, Weather, Trail.

A missing fact stays missing (NaN), never 0: a DayHike without elevation is
not a flat DayHike.

In [ ]:
import json
import os
import tempfile
from collections import Counter
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from dayhike import build_catalog, geometry_store_path_for
from dayhike.similarity import (
    CLIMATE_MONTH_COLUMNS,
    LANDCOVER_SHARE_COLUMNS,
    SURFACE_SHARE_COLUMNS,
    altitude_facts,
    effort_facts,
    landscape_facts,
    load_region,
    profile_facts,
    profile_lockstep,
    similarity_facts_path_for,
    surface_class,
    surface_class_from_tag,
    trail_facts,
    weather_facts,
)

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")
import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
if (
    local.is_file()
    and geometry_store_path_for(local).is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
):
    catalog_path = local
else:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(fixture, catalog_path)

REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
vertices, amenities, dayhikes, ways = load_region(
    catalog_path, lat_range=REGION_LAT, lon_range=REGION_LON
)
print("catalog", catalog_path)
for name, frame in (
    ("vertices", vertices),
    ("amenities", amenities),
    ("dayhikes", dayhikes),
    ("ways", ways),
):
    print(f"region {name}\t{len(frame)}")
print("name_origin", dayhikes["name_origin"].value_counts().to_dict())

## Effort

How much work the day is. Three facts per DayHike:

- `length_m`: the Catalog length of the walked vertex chain.
- `elevation_gain_m`: metres climbed along the chain. OSM way order is
  arbitrary, so a point-to-point DayHike stored "downhill" would show almost no
  climb. The fact is the larger of the climb and the descent along the chain,
  which is the same in both directions. On a loop the two are equal. Vertices
  without `ele` are skipped (the Catalog does the same); no `ele` at all leaves
  the gain missing.
- `difficulty_rank`: Difficulty as the SAC rank 1–6 (`hiking` 1 …
  `difficult_alpine_hiking` 6), from the strictest `sac_scale` on member ways.
  Missing when no member way is tagged, which is common.

In [ ]:
effort = effort_facts(dayhikes, vertices)
print(effort.describe().round(1).to_string())
print("effort missing share")
print(effort.isna().mean().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(effort["length_m"].dropna() / 1000, bins=40, color="#0f766e")
axes[0].set_xlabel("length (km)")
axes[1].hist(effort["elevation_gain_m"].dropna(), bins=40, color="#c2410c")
axes[1].set_xlabel("elevation gain (m)")
ranks = effort["difficulty_rank"].value_counts().sort_index()
axes[2].bar(ranks.index.astype(int).astype(str), ranks.to_numpy(), color="#7c3aed")
axes[2].set_xlabel("Difficulty rank (SAC)")
for ax in axes:
    ax.set_ylabel("DayHikes")
fig.suptitle("Effort facts in the region")
fig.tight_layout()
show_fig(fig)

## Altitude

How high the DayHike goes, apart from how much it climbs. Two facts, read from
the Catalog's `feature_vectors` (FABDEM when `sample-dem` has run, else OSM
`ele`):

- `highest_m`: the highest vertex.
- `lowest_m`: the lowest vertex. A coastal climb from sea level and a walk from
  a 1500 m pass can reach the same top.

Both are the same whichever way the DayHike is walked.

In [ ]:
altitude = altitude_facts(dayhikes)
print(altitude.describe().round(1).to_string())
print("altitude missing share")
print(altitude.isna().mean().round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = 40
ax.hist(altitude["lowest_m"].dropna(), bins=bins, alpha=0.6, color="#1d4ed8", label="lowest")
ax.hist(altitude["highest_m"].dropna(), bins=bins, alpha=0.6, color="#c2410c", label="highest")
ax.set_xlabel("elevation (m)")
ax.set_ylabel("DayHikes")
ax.set_title("Altitude facts in the region")
ax.legend()
fig.tight_layout()
show_fig(fig)

## Profile

The form of the elevation plot, apart from how high the DayHike goes and how
much it climbs. One fact per DayHike, `profile_shape`, its ProfileShape:

1. Cumulative distance along the vertex chain in stored order: haversine
   metres between consecutive vertices, over every vertex.
2. Elevation at 64 equal steps of the distance fraction 0 → 1, interpolated
   between the vertices that have `ele` (the ElevationProfile, resampled).
3. Subtract the lowest of the 64 values and divide by max(range, 50 m), so
   height is in [0, 1]. A DayHike that varies by less than 50 m is not
   stretched: 5 m of noise stays at most 0.1, and a flat walk looks flat.

Fewer than two vertices with `ele` leaves the ProfileShape missing. The shape
keeps the stored direction; `05a` compares it forward and reversed, so OSM's
way order does not matter.

The figure picks, for four contrasting forms, the region DayHike with an OSM
name whose ProfileShape is closest (`profile_lockstep`) to a drawn template: a single
summit (up and down), a traverse (one climb end to end), rolling (several
climbs) and flat. Each DayHike is drawn in its stored direction, so a traverse
may run downhill.

In [ ]:
profile = profile_facts(dayhikes, vertices)
shapes = profile["profile_shape"].dropna()
print(f"profile missing share\t{1 - len(shapes) / len(profile):.3f}")
print(f"profile range under 50 m share\t{shapes.map(lambda s: s.max() < 1).mean():.3f}")

fraction = np.linspace(0.0, 1.0, 64)
templates = {
    "single summit": np.interp(fraction, [0.0, 0.5, 1.0], [0.0, 1.0, 0.0]),
    "traverse": fraction,
    "rolling": (1 - np.cos(fraction * 8 * np.pi)) / 2,
    "flat": np.zeros(64),
}
hikes = dayhikes.set_index("id")
names = hikes["name"]
named = shapes[hikes.loc[shapes.index, "name_origin"] == "osm_name"]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, (form, template) in zip(axes, templates.items()):
    closest = named.map(lambda s: profile_lockstep(template, s)).idxmax()
    print(f"profile_example\t{form}\t{names[closest]}")
    ax.plot(fraction, template, color="#a8a29e", linestyle="--", label="template")
    ax.plot(fraction, shapes[closest], color="#0f766e", label="DayHike")
    ax.set_title(f"{form}\n{names[closest]}", fontsize=8)
    ax.set_xlabel("fraction of length")
axes[0].set_ylabel("ProfileShape height")
axes[0].legend(fontsize=8)
fig.tight_layout()
show_fig(fig)

## Landscape

What the DayHike looks like, apart from how hard or how high it is. Nine facts
per DayHike:

- Six LandCover shares (`tree_share`, `shrub_share`, `grass_share`,
  `bare_share`, `snow_share`, `other_share`). The first five are the Catalog's
  `feature_vectors` fractions (each GeometryStore vertex counts once). The
  sixth is the leftover, 1 − their sum, clipped at 0: WorldCover classes the
  Catalog does not keep (cropland, built-up, wetland, water) land there, so
  the six shares always sum to 1 and can be compared as a mix. When the
  Catalog fractions are missing (a hike outside the sampled rasters) all six
  are missing, not 0.
- `peak_summited`: a Peak (`natural=peak` amenity) within 50 m of the line.
- `viewpoint_along`: a Viewpoint (`tourism=viewpoint`) inside the 100 m
  Corridor.
- `peaks_around`: the number of Peaks within 2 km of the line.

**Corridor.** An amenity point is along a DayHike when it is within 100 m of
DayHikeGeometry, measured to the line and not only to the vertices (vertex
spacing reaches 120 m, and a way can have one segment of several hundred
metres). `corridor_amenities(vertices, amenities, kind=..., radius_m=...)`
inserts points along each segment, at most radius / 10 apart, and asks a
haversine BallTree for the amenity points near them. It returns one row per
(DayHike, point) with the distance to the line, so any amenity kind and any
radius work. The Water and Facilities Facets reuse it. All three radii here
(50 m, 100 m, 2 km) use the same helper, so the line is read the same way
whichever way the DayHike is walked.

Peaks and Viewpoints are raw GeometryStore amenities, not Catalog
AmenityFacts: they are looked up around the line, not the Trailhead.

In [ ]:
landscape = landscape_facts(dayhikes, vertices, amenities)
print(landscape.describe().round(3).to_string())
print("landscape missing share")
print(landscape.isna().mean().round(3).to_string())
print(f"peak_summited_hikes\t{int(landscape['peak_summited'].sum())}\tof\t{len(landscape)}")
print(f"viewpoint_along_hikes\t{int(landscape['viewpoint_along'].sum())}\tof\t{len(landscape)}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
shares = landscape[list(LANDCOVER_SHARE_COLUMNS)].dropna()
colors = ["#15803d", "#84cc16", "#facc15", "#a8a29e", "#38bdf8", "#e7e5e4"]
left = 0.0
for column, color in zip(LANDCOVER_SHARE_COLUMNS, colors):
    width = shares[column].mean()
    axes[0].barh([0], [width], left=left, color=color, edgecolor="white", label=column[:-6])
    left += width
axes[0].set_xlim(0, 1)
axes[0].set_yticks([])
axes[0].set_xlabel("mean share of the DayHike")
axes[0].set_title("LandCover mix (six shares)", fontsize=10)
axes[0].legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.3))
axes[1].hist(shares["tree_share"], bins=20, color="#15803d")
axes[1].set_xlabel("tree share")
axes[1].set_ylabel("DayHikes")
axes[2].hist(landscape["peaks_around"], bins=30, color="#7c3aed")
axes[2].set_xlabel("Peaks within 2 km")
axes[2].set_ylabel("DayHikes")
fig.suptitle("Landscape facts in the region")
fig.tight_layout()
show_fig(fig)

## Weather

What weather the DayHike is likely to have, and how much sun its slopes get.
Forty facts per DayHike:

- The ClimateNormal regime, read from the Catalog's `feature_vectors`
  (WorldClim 2.1, 1970–2000, averaged along DayHikeGeometry):
  `temperature_c` (June–August mean, °C), `precipitation_mm` (mean monthly
  precipitation over the year, mm) and `solar_radiation` (June–August mean,
  kJ m⁻² day⁻¹). This is the slice used when the Hiker names no QueryMonth.
- The twelve months, `temperature_c_01` … `solar_radiation_12`, read from the
  Catalog's `climate_months`. `05a` has one `QUERY_MONTH` setting that swaps
  the regime for one month's three values (ADR-0010).
- `southness`: SolarAspect, how south-facing the slopes along the DayHike are.
  It is terrain read from the DEM, not weather (shade from trees is
  LandCover). It sits in this Facet because it says how much sun the hike
  gets.

**SolarAspect rule** (`solar_aspect_southness`, FABDEM 1° tiles in `data/dem/`,
ADR-0008):

1. About 32 vertices per DayHike: every k-th vertex counted from the nearer
   chain end (k = ⌈vertices / 32⌉), so the same vertices are read whichever
   way the DayHike is walked.
2. FABDEM is read at the vertex and 2 arc-seconds east and north of it. That
   is two FABDEM cells (about 62 m north, 44 m east at 45° N), so the offset
   always reads a different cell, at a known distance.
3. The rise to the east and to the north give the slope and the downslope
   direction. Southness = (1 − the north part of the downslope direction) / 2:
   1 due south, 0 due north, 0.5 east or west, 0.85 south-east or south-west.
4. Points with a slope under 5° are left out. A valley floor or a broad
   ridge top faces no way, and a metre or two of FABDEM noise over 44–62 m
   would give it a random direction. The DayHike's southness is the mean over
   its sloped points. With none (a flat walk) or with no covering FABDEM tile
   (the committed fixture has none), southness is missing, not 0.5.

Notebook `04`'s prototype scored (1 + cos(aspect − π)) / 2 with the aspect
measured from east, which gives 1 for a west-facing slope and 0.5 for a
south-facing one. It is not used here. The rule above is tested on synthetic
north-, south-, east- and west-facing ramp rasters in `tests/test_similarity.py`.

On a mostly flat walk the few sloped points (an embankment, a river bank)
decide southness, so its value says little; it is still a value, because
some slope was found. The maps therefore show the three most north-facing and
the three most south-facing named DayHikes of at least 3 km with at least
500 m between their lowest and highest point, on a topographic basemap (the
line alone when offline).

In [ ]:
dem_dir = catalog_path.parent / "dem"
weather = weather_facts(dayhikes, vertices, dem_dir)
WEATHER_REGIME = ["temperature_c", "precipitation_mm", "solar_radiation", "southness"]
print("dem", dem_dir)
print(weather[WEATHER_REGIME].describe().round(2).to_string())
print("weather missing share")
print(weather[WEATHER_REGIME].isna().mean().round(3).to_string())
print(f"climate_months_missing_share\t{weather[list(CLIMATE_MONTH_COLUMNS)].isna().any(axis=1).mean():.3f}")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
labels = [
    "June–August temperature (°C)",
    "mean monthly precipitation (mm)",
    "June–August solar radiation (kJ m⁻² day⁻¹)",
    "southness (0 north, 1 south)",
]
for ax, column, label, color in zip(axes, WEATHER_REGIME, labels, ["#c2410c", "#1d4ed8", "#ca8a04", "#7c3aed"]):
    ax.hist(weather[column].dropna(), bins=30, color=color)
    ax.set_xlabel(label, fontsize=8)
axes[0].set_ylabel("DayHikes")
fig.suptitle("Weather facts in the region (ClimateNormal regime and SolarAspect)")
fig.tight_layout()
show_fig(fig)

hikes = dayhikes.set_index("id")
facing = weather[["southness"]].join(altitude).join(hikes[["name", "name_origin", "length_m", "geometry"]])
facing = facing[
    (facing["name_origin"] == "osm_name")
    & (facing["length_m"] >= 3_000)
    & (facing["highest_m"] - facing["lowest_m"] >= 500)
].dropna(subset=["southness"])
if facing.empty:
    print("southness missing everywhere: no FABDEM tile covers these DayHikes")
else:
    examples = pd.concat([facing.nsmallest(3, "southness"), facing.nlargest(3, "southness")])
    fig, axes = plt.subplots(1, len(examples), figsize=(3 * len(examples), 3.2))
    for ax, (dayhike_id, row) in zip(np.atleast_1d(axes), examples.iterrows()):
        print(f"southness_example\t{row['name']}\t{row['southness']:.2f}")
        one = gpd.GeoDataFrame([row], geometry="geometry", crs=dayhikes.crs).to_crs(epsg=3857)
        one.plot(ax=ax, color="#1d4ed8" if row["southness"] < 0.5 else "#c2410c", linewidth=2)
        try:
            import contextily as cx

            cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
        except Exception as error:  # offline: the line alone is still the map
            print("basemap skipped", type(error).__name__)
        ax.set_title(f"{row['name']}\nsouthness {row['southness']:.2f}", fontsize=8)
        ax.set_axis_off()
    fig.suptitle("Most north-facing (blue) and most south-facing (orange) named DayHikes")
    fig.tight_layout()
    show_fig(fig)

## Trail

What the DayHike walks on, apart from where it is. Five facts per DayHike,
from its member ways in the chain (`in_chain = 1`; leftover members are not
walked):

- Four SurfaceMix shares (`paved_share`, `gravel_share`, `natural_share`,
  `rock_share`): the length fraction of each class, length being the haversine
  along the way's nodes. They sum to 1 over the ways that could be classified.
- `tarmac_approach_m`: TarmacApproach. Start at the chain end nearest the
  Catalog Trailhead and add up the length of consecutive paved ways until the
  first way that is not paved. It is 0 when the first way is unpaved, and the
  whole classified length when every way is paved. The ways are put in walking
  order from the vertex chain (member order is not walking order: ingest can
  glue a member onto the start), and the end is chosen from the Trailhead, so
  the walking direction does not matter.

**Classifying a way.** The `surface` tag decides when it is one of these
(first `;` token, lowercased):

| Class | `surface` values |
| --- | --- |
| paved | asphalt, concrete (and `concrete:*`), paving_stones, sett, cobblestone, paved |
| gravel | compacted, gravel, fine_gravel, unpaved, pebblestone |
| natural | ground, dirt, earth, grass, mud, sand, woodchips |
| rock | rock, bare_rock, stone |

A way with no `surface`, or an unlisted one (`wood`, `metal`, ...), is
classified from its `highway`: roads (motorway down to residential, service,
living_street, pedestrian, cycleway, unclassified, and their `_link` forms)
are paved; a `track` follows its `tracktype` (grade1 paved, grade2-3 gravel,
grade4-5 natural, none gravel); `path`, `footway`, `steps` and `bridleway` are
natural; `via_ferrata` is rock. Any other way is left out of the denominator
(and ends the TarmacApproach). A DayHike with no classified way has all five
facts missing, not 0.

In [ ]:
trail = trail_facts(dayhikes, vertices, ways)
print(trail.describe().round(3).to_string())
print("trail missing share")
print(trail.isna().mean().round(3).to_string())

walked = ways[ways["in_chain"] == 1].drop_duplicates("way_id")
bags = [json.loads(tags) for tags in walked["tags"]]
by_tag = sum(surface_class_from_tag(bag) is not None for bag in bags)
by_fallback = sum(
    surface_class_from_tag(bag) is None and surface_class(bag) is not None for bag in bags
)
print(
    f"ways_classified\tby_tag={by_tag}\tby_highway_fallback={by_fallback}\t"
    f"left_out={len(bags) - by_tag - by_fallback}\tof\t{len(bags)}"
)
print(
    "fallback_highways",
    Counter(
        bag.get("highway")
        for bag in bags
        if surface_class_from_tag(bag) is None and surface_class(bag) is not None
    ).most_common(8),
)
print(f"tarmac_zero_share\t{(trail['tarmac_approach_m'] == 0).mean():.3f}")

SURFACE_COLORS = {"paved": "#475569", "gravel": "#d97706", "natural": "#15803d", "rock": "#a8a29e"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
mean_mix = trail[list(SURFACE_SHARE_COLUMNS)].mean()
left = 0.0
for kind, column in zip(SURFACE_COLORS, SURFACE_SHARE_COLUMNS):
    axes[0].barh([0], [mean_mix[column]], left=left, color=SURFACE_COLORS[kind], edgecolor="white", label=kind)
    left += mean_mix[column]
axes[0].set_xlim(0, 1)
axes[0].set_yticks([])
axes[0].set_xlabel("mean share of the DayHike")
axes[0].set_title("SurfaceMix (four shares)", fontsize=10)
axes[0].legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3))
axes[1].hist(trail["paved_share"].dropna(), bins=20, color=SURFACE_COLORS["paved"])
axes[1].set_xlabel("paved share")
axes[1].set_ylabel("DayHikes")
axes[2].hist(trail["tarmac_approach_m"].dropna().clip(upper=6_000) / 1_000, bins=30, color="#7c3aed")
axes[2].set_xlabel("TarmacApproach (km, clipped at 6)")
axes[2].set_ylabel("DayHikes")
fig.suptitle("Trail facts in the region")
fig.tight_layout()
show_fig(fig)

# One DayHike with a road approach and three or more surfaces, ways coloured by class.
classes_present = (trail[list(SURFACE_SHARE_COLUMNS)] > 0.05).sum(axis=1)
candidates = trail[(trail["tarmac_approach_m"] > 0) & (classes_present >= 3)]
example_id = int((candidates if len(candidates) else trail.dropna()).index[0])
example = ways[(ways["dayhike_id"] == example_id) & (ways["in_chain"] == 1)].copy()
example["surface_class"] = [surface_class(json.loads(tags)) or "unknown" for tags in example["tags"]]
example = example[example.geometry.notna()]
print(f"example_dayhike\t{example_id}\t{dayhikes.set_index('id').at[example_id, 'name']}")
print(trail.loc[[example_id]].round(3).to_string())
fig, ax = plt.subplots(figsize=(6, 5))
for kind, color in {**SURFACE_COLORS, "unknown": "#dc2626"}.items():
    part = example[example["surface_class"] == kind]
    if len(part):
        part.plot(ax=ax, color=color, linewidth=3, label=kind)
ax.legend(fontsize=8)
ax.set_title(f"Ways of DayHike {example_id} by surface class", fontsize=10)
fig.tight_layout()
show_fig(fig)

## Similarity facts table

One row per region DayHike, indexed by DayHike id: `name`, `name_origin`, the
Trailhead and the DayHike line (for maps in `05a`), then every Facet's facts.
It is written as a pandas pickle next to the Catalog,
`{catalog}.similarity_facts.pkl` (`data/catalog.similarity_facts.pkl` for the
local Catalog). It is regenerable and gitignored (`*.similarity_facts.pkl`).

In [ ]:
facts = gpd.GeoDataFrame(
    dayhikes.set_index("id")[
        ["name", "name_origin", "trailhead_latitude", "trailhead_longitude", "geometry"]
    ]
    .join(effort)
    .join(altitude)
    .join(profile)
    .join(landscape)
    .join(weather)
    .join(trail),
    geometry="geometry",
    crs=dayhikes.crs,
)
facts_path = similarity_facts_path_for(catalog_path)
facts.to_pickle(facts_path)
print("similarity_facts", facts_path)
print(f"similarity_facts_rows\t{len(facts)}")
print("columns", list(facts.columns))